In [7]:
%matplotlib tk 
# 로컬 주피터 매직 커맨드 

import torch
import platform
import textwrap
import re
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
from wordcloud import WordCloud
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

In [8]:
# 모델과 토크나이저 불러오기
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print("모델을 로드하는 중입니다...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
quant_config = QuantoConfig(weights="int8") # 8비트(INT8) 양자화 설정 객체 생성

# 모델 로드 시 양자화 설정을 주입하여 압축된 상태로 메모리에 올림
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

모델을 로드하는 중입니다...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [9]:
# 워드클라우드 생성 함수 정의
def run_wordcloud(model, tokenizer, prompt, top_n=30):
    # 한글 폰트 설정
    system_name = platform.system()
    if system_name == 'Windows':
        font_path = 'c:/Windows/Fonts/malgun.ttf'
    elif system_name == 'Darwin':
        font_path = '/System/Library/Fonts/AppleGothic.ttf'
    else:
        font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    font_prop = FontProperties(fname=font_path)

    # 2. 화면 분할
    fig, (ax_text, ax_wc) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 4]})
    fig.canvas.manager.set_window_title('LLM과 함께 글쓰기')
    
    # 모델을 감성적으로 만드는 숨겨진 프롬프트
    hidden_prefix = (
        "[당신은 감수성이 풍부하고 세상을 아름답게 바라보는 시인입니다. "
        "서정적이고 감동적인 시어로 다음 문장을 이어가세요.]\n\n"
    )
    
    state = {"text": prompt}

    def draw_current_state():
        # [상단 영역] 완성된 문장 표시 (숨겨진 프롬프트는 보여주지 않음)
        ax_text.clear()
        ax_text.axis('off')
        
        wrapped_text = textwrap.fill(state["text"], width=70)
        bbox_style = dict(boxstyle="round,pad=1", facecolor="#fff0f5", edgecolor="#ffb6c1", linewidth=2)
        
        ax_text.text(0.5, 0.5, f"현재 문장:\n\n{wrapped_text}", 
                     ha='center', va='center', fontsize=16, fontproperties=font_prop, 
                     bbox=bbox_style, linespacing=1.5)

        # [하단 영역] 워드클라우드 표시
        ax_wc.clear()
        ax_wc.axis('off')
        ax_wc.set_xlim(0, 800)
        ax_wc.set_ylim(400, 0) 
        
        # 모델에 숨겨진 지시문과 현재 문장을 합쳐서 전달
        full_text_for_model = hidden_prefix + state["text"]
        
        inputs = tokenizer(full_text_for_model, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
        
        logits = outputs.logits[0, -1, :]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        top_probs, top_indices = torch.topk(probs, top_n)
        
        token_data = {}
        for i in range(top_n):
            token_id = top_indices[i].item()
            prob = top_probs[i].item()
            
            # 특수 토큰(BOS, EOS 등)을 제외하고 디코딩
            raw_str = tokenizer.decode([token_id], skip_special_tokens=True)
            display_str = raw_str.strip()
            
            # 공백, 빈 문자열이 포함된 경우 스킵
            if not display_str:
                continue
                
            if display_str not in token_data:
                token_data[display_str] = {"prob": prob, "raw": raw_str}
            else:
                token_data[display_str]["prob"] += prob
                
        freqs = {k: v["prob"] for k, v in token_data.items()}
        
        # 상위 토큰이 모두 필터링되어 freqs가 비었을 경우의 방어 로직
        if not freqs:
            fallback_text = "[더 이상 예측할 단어가 없습니다]"
            freqs = {fallback_text: 1.0}
            token_data[fallback_text] = {"prob": 1.0, "raw": ""}
        
        # 워드클라우드 생성 (기본 컬러맵 viridis 자동 적용)
        wc = WordCloud(font_path=font_path, width=800, height=400, 
                       background_color='white', max_font_size=100)
        wc.generate_from_frequencies(freqs)
        
        for item in wc.layout_:
            (word, freq), font_size, (y, x), orientation, color_str = item
            rot_deg = 90 if orientation else 0
            
            # 'rgb(70, 52, 128)' 형태의 문자열을 튜플로 변환
            mpl_color = color_str
            if isinstance(color_str, str) and color_str.startswith('rgb'):
                match = re.match(r"rgb\((\d+),\s*(\d+),\s*(\d+)\)", color_str)
                if match:
                    # 0~255 사이의 값을 0~1 사이의 비율로 변환 (matplotlib 규칙)
                    mpl_color = tuple(int(v) / 255.0 for v in match.groups())
            
            # 변환된 mpl_color를 적용하여 텍스트 생성
            t = ax_wc.text(x, y, word, fontsize=font_size, color=mpl_color,
                           fontproperties=font_prop, rotation=rot_deg,
                           ha='left', va='top', picker=True)
            t.raw_str = token_data[word]["raw"]
            
        fig.canvas.draw()

    # 클릭 이벤트 처리
    def on_pick(event):
        text_obj = event.artist
        if hasattr(text_obj, 'raw_str'):
            selected_token = text_obj.raw_str
            # 빈 문자열 클릭 시 무시
            if not selected_token:
                return
                
            state["text"] += selected_token
            draw_current_state()

    fig.canvas.mpl_connect('pick_event', on_pick)
    
    draw_current_state()
    plt.tight_layout()
    plt.show()

In [10]:
# 함수 실행
input_word = input("단어를 입력해 글쓰기를 시작해 보세요. ")
run_wordcloud(model=model, tokenizer=tokenizer, prompt=input_word, top_n=30)

단어를 입력해 글쓰기를 시작해 보세요.  혼자
